### Example 5: Obstacle Control - Minimum Potential Energy Formulation

$$
\begin{aligned}
(\mathscr{P}) \quad \min_{u \in U_{ad}} \quad & J(y, u) = \frac{1}{2} \|y - z_d\|_{L^2(\Omega)}^2 + \frac{V}{2} \|u\|_{L^2(\Omega)}^2 \\
\text{subject to} \quad & y = \argmin_{w \in K} \mathscr{E}(w; u) \\
& K = \{ w \in H_0^1(\Omega) \mid w \geq 0 \text{ a.e. in } \Omega \}
\end{aligned}
$$

#### Potential Energy Functional
The energy functional $\mathscr{E}(w; u)$ is given by:

$$
\mathscr{E}(w; u) = \frac{1}{2}a(w,w) - \langle f + u, w \rangle
$$

where:
- $a(w,w) = \int_\Omega \nabla w \cdot \nabla w \, dx$ (Dirichlet energy)
- $\langle f + u, w \rangle = \int_\Omega (f + u)w \, dx$ (work done by external forces)

In [ ]:
from pathlib import Path
import sys

PACKAGE_ROOT = Path.cwd().resolve()
if not (PACKAGE_ROOT / "utils.py").is_file():
    raise FileNotFoundError("Run this notebook from the final_code package directory.")
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))

import matplotlib.pyplot as plt
import torch
from   torch.autograd import grad
from utils import *
from utils import net_v_res as net_u_res
from utils import visual_v_3d as visual_u_3d
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader,TensorDataset
from scipy.stats.qmc import LatinHypercube
from functools import partial
from torch.optim.lr_scheduler import LambdaLR
import os
import time 
from tqdm import tqdm
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
from IPython.display import clear_output
from datetime import datetime
ROOT = PACKAGE_ROOT / "outputs" / "example5"
DATA_ROOT = PACKAGE_ROOT / "data" / "example5"

In [ ]:
class Source8:
    def __init__(self, device='cpu', data_path=None):
        """
        Source terms with custom f function and data loaded from a MAT file
        
        Parameters:
            device: Computation device ('cpu' or 'cuda')
        """
        self.device = device
        self.nu = 0.5
        self.analystic = True  # Enable exact solution comparison
        
        if data_path is None:
            data_path = 'example3_data_modified.mat'
        data_path = os.path.join(DATA_ROOT, data_path)
        print(f"Loading data from: {data_path}")
        
        mat_data = sio.loadmat(data_path)
        
        # Extract coordinate data - these must exist
        self.x_coords = np.array(mat_data['x_coordinates']).flatten().astype(np.float64)
        self.y_coords = np.array(mat_data['y_coordinates']).flatten().astype(np.float64)
        
        nx = len(self.x_coords)
        ny = len(self.y_coords)
        print(f"Grid dimensions - x: {nx}, y: {ny}")
        
        # Extract required fields - these must exist
        if 'y_with_bc' not in mat_data:
            raise ValueError("The MAT file must contain 'y_with_bc' field")
        if 'psi_with_bc' not in mat_data:
            raise ValueError("The MAT file must contain 'psi_with_bc' field")
        
        self.y_values = np.array(mat_data['y_with_bc']).astype(np.float64)
        self.u_values = np.array(mat_data['psi_with_bc']).astype(np.float64)  # psi is the control u
        
        # Create interpolators for y and u on the full grid
        self.y_interpolator = RegularGridInterpolator(
            (self.y_coords, self.x_coords), self.y_values, 
            bounds_error=False, fill_value=None
        )
        
        self.u_interpolator = RegularGridInterpolator(
            (self.y_coords, self.x_coords), self.u_values, 
            bounds_error=False, fill_value=None
        )

        
        print("All required fields loaded and interpolators created successfully!")


    def compute_yd(self, x):
        """
        
        Parameters:
            x: Tensor of shape [batch_size, 2] containing (x,y) coordinates
            
        Returns:
            yd_values: Constant tensor with value 5
        """
        return 5 * torch.ones(x.shape[0], 1, device=x.device)

    def compute_f(self, x):
        """
        Compute custom source term f for Example 5:
        
        Parameters:
            x: Tensor of shape [batch_size, 2] containing (x,y) coordinates
        
        Returns:
            f_values: Source term values at points x
        """
        x1 = x[:, 0:1]
        x2 = x[:, 1:2]
        
        # Initialize f with default value 150
        f_values = 150.0 * torch.ones_like(x2, device=x.device)
        
        mask = (x2 > 0.25) & (x2 < 0.65)
        
        f_values[mask] = -100.0
        
        return f_values
    
    def compute_y_star(self, x):
        """
        Compute exact solution for state y at given points x from loaded data
        """
        x_np = x.detach().cpu().numpy()
        
        # For interpolation, points need to be in (y,x) order
        points = x_np[:, [1, 0]]  # Swap x and y coordinates to match interpolator
        
        try:
            y_exact = self.y_interpolator(points)
        except:
            # If interpolation fails, try without swapping
            y_exact = self.y_interpolator(x_np)
        
        return torch.tensor(y_exact, dtype=torch.float32, device=self.device).reshape(-1, 1)
    
    def compute_u_star(self, x):
        """
        Compute exact solution for control u at given points x from loaded data
        """
        x_np = x.detach().cpu().numpy()
        
        # For interpolation, points need to be in (y,x) order
        points = x_np[:, [1, 0]]  # Swap x and y coordinates to match interpolator
        
        try:
            u_exact = self.u_interpolator(points)
        except:
            # If interpolation fails, try without swapping
            u_exact = self.u_interpolator(x_np)
        
        return torch.tensor(u_exact, dtype=torch.float32, device=self.device).reshape(-1, 1)

## reload network

In [ ]:
class net_y_res(nn.Module):
    def __init__(self,source,psi=None, hidden=64, layers=3, init=True, activation='Tanh', hard_=None):
        super(net_y_res, self).__init__()
        
        self.activation = activation
        if activation == 'Tanh':
            self.activation = nn.Tanh()
        elif activation == 'Identity':
            self.activation = nn.Identity()
        elif activation == 'Square':
            self.activation = SquaredReLU()
        elif activation == 'Cubic':
            self.activation = CubicReLU()
        elif activation == 'SELU':
            self.activation = nn.SELU()
        elif activation == 'Swish':
            self.activation = nn.SiLU()
        elif activation == 'ReLU':
            self.activation = nn.ReLU()
        self.__dict__['_psi'] = psi
        self.source = source

        
        self.padding_size = hidden - 2
        
        # Residual blocks
        self.res_blocks = nn.ModuleList(
            [ResidualBlock(hidden, self.activation) for _ in range(layers - 1)]
        )
        
        self.output_layer = nn.Linear(hidden, 1)
        
        self.hard = hard_
            
        if init:
            self._initialize_weights()

    def _initialize_weights(self):
        nn.init.xavier_uniform_(self.output_layer.weight)
        nn.init.zeros_(self.output_layer.bias)
        
    def forward(self, x):        
        padded_x = torch.cat([x, torch.zeros(x.size(0), self.padding_size).to(x.device)], dim=1)
        
        # Apply hard constraint (if provided)
        x1 = padded_x[:, 0:1]
        x2 = padded_x[:, 1:2]
        
        if self.hard is None:
            hard = x1 * (1 - x1) * x2 * (1 - x2)
        else:
            hard = self.hard(x1, x2)
        
        out = padded_x
        for res_block in self.res_blocks:
            out = res_block(out)
        
        out = self.output_layer(out)
        
        # Apply hard constraint and ensure non-negative output
        out = out * hard
        if self._psi is not None:
            psi_val = self._psi(x)
            out = -torch.relu(psi_val-out)+psi_val
        
        return out


# Upper- and lower-level losses: objective and energy functional

In [ ]:
def J_u(source, net_y, net_psi, x):
    """
    Upper-level energy J(psi) = 1/2||y(psi)-y_d||^2 + (alpha/2)||psi||_{H1}^2
      = 1/2 * E[(y_pred - y_d)^2]
      + alpha/2 * E[ |∇psi|^2 ]
    Here E[...] approximates the integral by the mean.
    """
    # Make x differentiable to compute the gradient of psi
    x = x.clone().detach().requires_grad_(True)
    
    # 1) Obtain the state prediction y_pred(x) and obstacle prediction psi_pred(x)
    y_pred   = net_y(x)        # (batch,1)
    psi_pred = net_psi(x)      # (batch,1)
    
    # 2) Desired state y_d(x)
    y_d_val = source.compute_yd(x)  # (batch,1)
    
    # 3) Compute the gradient ∇psi
    dpsi_dx = grad(psi_pred.sum(), x, create_graph=True)[0]  # (batch,2)
    grad_norm_sq = (dpsi_dx**2).sum(dim=1, keepdim=True)    # |∇psi|^2, (batch,1)
    
    # 4) Approximate the integral by the mean
    term_y  = 0.5 * torch.mean((y_pred - y_d_val)**2)
    term_H1 = 0.5 * source.nu * torch.mean(grad_norm_sq )
    
    return term_y + term_H1

def J_l(source, net_y,net_u,x):
    """
    net_u:non-used
    The lower-level state y minimizes the potential energy:
    J(w, u) = 1/2*a(w,w) - <f,w>
    Use MSELoss to approximate the integral.
    """
    x = x.clone().detach().requires_grad_(True)
    x1 = x[:, 0:1]
    x2 = x[:, 1:2]
    x = torch.cat([x1, x2], 1)
    f_val = source.compute_f(x)
    y_pred  = net_y(x)
    y_x1 = grad(y_pred.sum(), x1, create_graph=True)[0]
    y_x2 = grad(y_pred.sum(), x2, create_graph=True)[0]

    # Compute a(w,w)
    a_val = y_x1**2 + y_x2**2
    linear_term = (f_val)*y_pred

    return torch.mean(0.5*a_val-linear_term)

# Relative errors

In [ ]:
def rel_l2_error_yu(net_y, net_u, source, device='cpu', resolution=101):
    """
    Compute the relative L2 errors of y and u on [0,1]×[0,1]:
       rel_l2_y = ||y_app - y_ref|| / ||y_ref||
       rel_l2_u = ||u_app - u_ref|| / ||u_ref||
       
    Return 0, 0 if source has no analytical solution.
    """
    has_analytic = hasattr(source, 'analystic') and source.analystic
    
    if not has_analytic:
        print("The source has no analytical solution; skipping error computation")
        return torch.tensor(0.0, device=device), torch.tensor(0.0, device=device)
    
    # Generate sample points on [0,1]
    x = torch.linspace(0, 1, resolution, device=device)
    X, Y = torch.meshgrid(x, x, indexing='ij')
    
    xy = torch.cat((X.reshape(-1, 1), Y.reshape(-1, 1)), dim=1)
    
    # Sample interior points
    X, Y = torch.meshgrid(x[1:-1], x[1:-1], indexing='ij')
    xy_interior = torch.cat((X.reshape(-1, 1), Y.reshape(-1, 1)), dim=1)
    
    # Compute the reference (exact) solutions
    with torch.no_grad():
        y_star = source.compute_y_star(xy)  # y_star is the exact solution at all points
        u_star = source.compute_u_star(xy_interior)  # u_star is the exact solution at interior points
    
    y_app = net_y(xy)
    u_app = net_u(xy_interior)
    
    num_y = torch.sum((y_app - y_star) ** 2)
    den_y = torch.sum(y_star ** 2)
    num_u = torch.sum((u_app - u_star) ** 2)
    den_u = torch.sum(u_star ** 2)
    
    # Avoid division by zero; handle a zero reference solution as needed
    rel_l2_y = torch.sqrt(num_y / (den_y + 1e-30))
    rel_l2_u = torch.sqrt(num_u / (den_u + 1e-30))
    
    print(f"Relative L2 error for y: {rel_l2_y.item():.4e}")
    print(f"Relative L2 error for u: {rel_l2_u.item():.4e}")
    
    return rel_l2_y, rel_l2_u


# Random sampler L

In [ ]:
class RandomDataLoader:
    """
    Random data loader
    """
    def __init__(self, batch_size=512, steps_per_epoch=4, dims=2, device='cpu',
                 dual_sampling=False):
        """
        Initialize a fast random data loader

        Parameters:
            batch_size (int): Target total batch size
            steps_per_epoch (int): Number of batches per epoch
            dims (int): Input dimension (default 2, for x1 and x2)
            device (str): Device for data generation ('cpu', 'cuda:0', etc.)
            dual_sampling (bool): Whether to generate two independent samples for the same iteration
        """
        self.steps_per_epoch = steps_per_epoch
        self.dims = dims
        self.device = device
        self.current_step = 0
        
        # Compute the number of interior points so the total is close to the target batch_size
        self.interior_points = max(1, batch_size)
        
        self.batch_size = self.interior_points
        self.dual_sampling = dual_sampling
 
    def __iter__(self):
        """Return the iterator itself"""
        self.current_step = 0
        return self
    
    def __next__(self):
        """Generate a complete batch on each call"""
        if self.current_step < self.steps_per_epoch:
            self.current_step += 1
            
            if self.dual_sampling:
                # Use shared points for y/theta and resample u independently
                yz_points = torch.rand(self.interior_points, self.dims, device=self.device)
                u_points = torch.rand(self.interior_points, self.dims, device=self.device)
                return yz_points, u_points
            
            single_points = torch.rand(self.interior_points, self.dims, device=self.device)
            return (single_points,)
        else:
            # Reset and raise StopIteration when the epoch step limit is reached
            self.current_step = 0
            raise StopIteration
    
    def __len__(self):
        """Return the number of batches per epoch"""
        return self.steps_per_epoch

# Trainer

In [ ]:
class Trainer:
    def __init__(
        self,
        u: nn.Module,
        y: nn.Module,
        theta: nn.Module,
        source: Source,
        train_loader: DataLoader,
        optimizer_u: torch.optim.Optimizer,
        optimizer_y: torch.optim.Optimizer,
        optimizer_theta: torch.optim.Optimizer,
        scheduler_u: torch.optim.lr_scheduler.LambdaLR,
        scheduler_y: torch.optim.lr_scheduler.LambdaLR,
        scheduler_theta: torch.optim.lr_scheduler.LambdaLR,
        gamma: float = 10.0,
        p: float = 0.5,
        device: torch.device = torch.device("cpu"),
        num_epochs: int = 10000,
        weight: list = [1, 1],
        save: bool = False,
        lr: float = 1e-3,
    ):    
        """
        Args:
            u (nn.Module): Control u(x).
            y (nn.Module): State y(x).
            theta (nn.Module): theta(x).
            source (Source): Source instance.
            train_loader (DataLoader): DataLoader.
            optimizer (torch.optim.Optimizer): Optimizer.
            scheduler (torch.optim.lr_scheduler.LambdaLR): Learning rate scheduler.
            gamma (float): weight of Moreau envelope.
            p (float): parameter of penalty.
            0 <= p < 1.
            device (torch.device): Training device.
            num_epochs (int): Maximum number of training epochs.
            weight (list): weight of loss.
            save (bool): Whether to save the models.
        """
    
        self.u = u.to(device)
        self.y = y.to(device)
        self.theta = theta.to(device)
        self.source = source
        self.train_loader = train_loader
        self.optimizer_u = optimizer_u
        self.optimizer_y = optimizer_y
        self.optimizer_theta = optimizer_theta
        self.scheduler_u = scheduler_u
        self.scheduler_y = scheduler_y
        self.scheduler_theta = scheduler_theta
        self.gamma = gamma
        self.p = p
        self.device = device
        self.num_epochs = num_epochs
        self.weight = weight
        self.save = save
        self.num = 0
        self.timestart = 0
        self.record_u = torch.zeros(1).to(self.device)
        self.record_l = torch.zeros(1).to(self.device)
        self.record_ortho = torch.zeros(1).to(self.device)
        self.record_non_neg = torch.zeros(1).to(self.device)
        self.lr = lr
        self.c0 = 1
        self.nu = source.nu
        self.file_suffix = (
            f"stepunchanged{datetime.now().strftime('%Y%m%d')}udecay_gamma_{self.gamma}"
            f"_warmup_{warmup}_cycle_{cycle_factor}_len_{cycle_length}"
            f"_batch_size{batch_size}_{self.num_epochs}"
            f"_p_{self.p}_lr_{self.lr}_nu_{self.nu}_u"
        )
        if save == True:
            os.makedirs(f'{ROOT}/{self.file_suffix}', exist_ok=True)
        if save == True:
            self.loss_csv_file = f'{ROOT}/{self.file_suffix}/loss.csv'
            with open(self.loss_csv_file, 'w') as f:
                f.write("epoch,time,J_u_loss,J_l_loss,diff_net\n")
            self.norm_csv_file = f'{ROOT}/{self.file_suffix}/norm.csv'
            with open(self.norm_csv_file, 'w') as f:
                f.write("epoch,time,rel_l2_error_y,rel_l2_error_u\n")


    def diff_net(self, net_y, net_theta):
        """
        Compute the difference between the parameters of two networks, considering only linear-layer weights and biases.
        """
        res = 0
        for para1, para2 in zip(net_y.parameters(), net_theta.parameters()):
            res += torch.nn.MSELoss(reduction='sum')(para1, para2)
        return res
    
    def getloss_u(self,source,net_y, net_u, x):
        if not x.requires_grad:
            x = x.detach().requires_grad_(True)
        return self.weight[0]*J_u(source, net_y, net_u, x)
    
    def getloss_l(self, source, net_y, net_u, x):
        if not x.requires_grad:
            x = x.detach().requires_grad_(True)
        return self.weight[1] * J_l(source, net_y, net_u, x)
    
    def train_one_epoch(self, epoch, timestart):
        self.u.train()
        self.y.train()
        self.theta.train()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
        ck = 0.2*(epoch)**self.p
        inv_ck = 1.0 / ck if ck > 0 else 0.0
        for batch in self.train_loader:
            if isinstance(batch, (tuple, list)):
                if len(batch) == 2:
                    x_ytheta_raw, x_u_raw = batch
                else:
                    x_ytheta_raw = batch[0]
                    x_u_raw = batch[0]
            else:
                x_ytheta_raw = batch
                x_u_raw = batch
            
            x_ytheta = x_ytheta_raw.detach().to(self.device)
            x_theta = x_ytheta.clone().requires_grad_(True)
            x_y = x_ytheta.clone().requires_grad_(True)
            x_u = x_u_raw.detach().to(self.device).requires_grad_(True)
            
            with torch.set_grad_enabled(True):
                theta_params = list(self.theta.parameters())
                y_params = list(self.y.parameters())
                u_params = list(self.u.parameters())
                
                # 1. Use the shared batch for theta and add the prox term explicitly
                loss_theta_e = self.getloss_l(self.source, self.theta, self.u, x_theta)
                grad_theta_e = torch.autograd.grad(loss_theta_e, theta_params, retain_graph=False)
                theta_total_grads = []
                for g_e, p_theta, p_y in zip(grad_theta_e, theta_params, y_params):
                    prox_term = (p_theta.detach() - p_y.detach()) * (1.0 / self.gamma)
                    theta_total_grads.append(g_e + prox_term)
                self._apply_custom_grads(self.theta, self.optimizer_theta, theta_total_grads)
                
                # 2. Compute the y gradient using the shared batch
                loss_upper_y = self.getloss_u(self.source, self.y, self.u, x_y)
                loss_lower_y = self.getloss_l(self.source, self.y, self.u, x_y)
                h_j_y = torch.autograd.grad(loss_upper_y, y_params, retain_graph=True)
                h_e_y = torch.autograd.grad(loss_lower_y, y_params, retain_graph=False)
                y_total_grads = []
                for g_j, g_e, p_y, p_theta in zip(h_j_y, h_e_y, y_params, theta_params):
                    prox_term = (p_theta.detach() - p_y.detach()) * (1.0 / self.gamma)
                    y_total_grads.append(inv_ck * g_j + g_e + prox_term)
                    
                self._apply_custom_grads(self.y, self.optimizer_y, y_total_grads)
                # 3. Compute the u gradient using the resampled batch
                loss_upper_u = self.getloss_u(self.source, self.y, self.u, x_u)
                loss_lower_u = self.getloss_l(self.source, self.y, self.u, x_u)
                loss_lower_theta_u = self.getloss_l(self.source, self.theta, self.u, x_u)
                h_j_u = torch.autograd.grad(loss_upper_u, u_params, retain_graph=True)
                h_e_u = torch.autograd.grad(loss_lower_u, u_params, retain_graph=True)
                h_e_u_theta = torch.autograd.grad(loss_lower_theta_u, u_params, retain_graph=False)
                u_total_grads = []
                for g_j_u, g_e_u, g_e_u_theta in zip(h_j_u, h_e_u, h_e_u_theta):
                    u_total_grads.append(inv_ck * g_j_u + g_e_u - g_e_u_theta)

                
                self._apply_custom_grads(self.u, self.optimizer_u, u_total_grads)


        self.scheduler_u.step()
        self.scheduler_y.step()
        self.scheduler_theta.step()

        diff_net_value = self.diff_net(self.y, self.theta).item()

        if (epoch-1) % 10 ==0:
            # Evaluate losses on a uniform grid
            grid_size = 50  # Number of points per dimension
            x1 = torch.linspace(0, 1, grid_size, device=self.device)
            x2 = torch.linspace(0, 1, grid_size, device=self.device)
            X1, X2 = torch.meshgrid(x1, x2, indexing='ij')
            grid_points = torch.stack([X1.flatten(), X2.flatten()], dim=1).requires_grad_(True)
            self.record_u = J_u(self.source, self.y, self.u, grid_points)
            self.record_l = J_l(self.source, self.y, self.u, grid_points)
            with open(self.loss_csv_file, 'a') as f:
                f.write(f"{epoch},{time.time()-timestart},{self.record_u.item()},{self.record_l.item()},{diff_net_value}\n")

            if epoch % 10 == 0 and torch.cuda.is_available():
                torch.cuda.empty_cache()

        return self.record_u.item(), self.record_l.item(), diff_net_value
    
    def _update_model(self, model, optimizer, loss):
        optimizer.zero_grad()
        grad_params = torch.autograd.grad(loss, model.parameters())
        for param, g in zip(model.parameters(), grad_params):
            param.grad = g
        optimizer.step()
    
    def _apply_custom_grads(self, model, optimizer, grads):
        optimizer.zero_grad()
        for param, grad in zip(model.parameters(), grads):
            param.grad = grad
        optimizer.step()
    
    def validate(self,epoch):
        self.u.eval()
        self.y.eval()
        plt.close('all')
        rel_l2_error_y, rel_l2_error_u = rel_l2_error_yu(self.y, self.u, self.source, device=self.device)
        with open(self.norm_csv_file, 'a') as f:
            f.write(f"{epoch},{time.time()-self.timestart},{rel_l2_error_y.item()},{rel_l2_error_u.item()}\n")
            
        with torch.no_grad():
            if self.save == True:
                yfile = f'{ROOT}/{self.file_suffix}/y_net_{epoch}.png'
                ufile = f'{ROOT}/{self.file_suffix}/u_net_{epoch}.png'
                xifile = f'{ROOT}/{self.file_suffix}/xi_net_{epoch}.png'
            else:
                yfile = None
                ufile = None
                xifile = None
            visual_y_3d(self.source, self.y, file_name=yfile)
            visual_u_3d(self.source, self.u, file_name=ufile)
        

    def fit(self):
        pbar = tqdm(range(1, self.num_epochs + 2), ascii=True, dynamic_ncols=True)
        self.timestart = time.time()
        clear_output(wait=True)
        for epoch in pbar:
            J_u_loss, J_l_loss, diff_net_value = self.train_one_epoch(epoch, self.timestart)
            pbar.set_postfix(
                J_u_loss=J_u_loss,
                J_l_loss=J_l_loss,
                diff_net=diff_net_value
            )
            
            if (epoch-1) % 500 == 0:
                self.validate(epoch)

# Set the device here

In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
source = Source8(device=device)
set_seed(123)

# Hyperparameter settings

In [ ]:
# hyperparameters
lr = 1e-3
warmup = 1000
cycle_factor=0.8
cycle_length=1000
num_epochs=10000
weight = [1, 1]
save= True
batch_size = 512

## Initialize and load existing pretrained models

In [ ]:
dataloader = RandomDataLoader(
    batch_size=batch_size,           # Target batch size 
    steps_per_epoch=4,       # Number of batches per epoch
    device=device,
    dual_sampling=True  # Sample y/theta in the same batch and u separately
)

u = net_u_res(source=source,hidden=16,layers=4,activation = 'Swish',hard_=hard_constraint_2d).to(device)
y = net_y_res(psi=u,source=source,hidden=16,layers=4,activation = 'Swish',hard_=hard_constraint_2d).to(device)
theta = net_y_res(psi=u,source=source,hidden=16,layers=4,activation = 'Swish',hard_=hard_constraint_2d).to(device)
# Load the pre-trained models
y.load_state_dict(torch.load(DATA_ROOT / 'pretrain_y_energy_final.pth', map_location=device))
# Initialize theta with the same weights as y
theta.load_state_dict(y.state_dict())

## Training

In [ ]:
optimizer_y = torch.optim.Adam(y.parameters(), lr=lr, betas=(0.9, 0.999))
optimizer_u = torch.optim.Adam(u.parameters(), lr=lr)
optimizer_theta = torch.optim.Adam(theta.parameters(), lr=lr, betas=(0.9, 0.999))
scheduler_y = LambdaLR(optimizer_y,partial(lr_lambda,warmup=warmup,cycle_factor=cycle_factor,cycle_length=cycle_length))
scheduler_u = LambdaLR(optimizer_u,partial(lr_lambda,warmup=warmup,cycle_factor=cycle_factor,cycle_length=cycle_length))
scheduler_theta = LambdaLR(optimizer_theta,partial(lr_lambda,warmup=warmup,cycle_factor=cycle_factor,cycle_length=cycle_length))
trainer = Trainer(
    u=u,
    y=y,
    theta=theta,
    source=source,
    train_loader=dataloader,
    optimizer_u=optimizer_u,
    optimizer_y=optimizer_y,
    optimizer_theta=optimizer_theta,
    scheduler_u=scheduler_u,
    scheduler_y=scheduler_y,
    scheduler_theta=scheduler_theta,
    gamma=50.0,
    p=0.2,
    device=device,
    num_epochs=num_epochs,
    weight=weight,
    save=save,
    lr = lr,
)
trainer.fit()

In [ ]:
import os

# Save the final models in the models directory under the same subfolder as loss.csv and norm.csv
model_dir = os.path.join(ROOT, trainer.file_suffix, "models")
os.makedirs(model_dir, exist_ok=True)
torch.save(y.state_dict(), os.path.join(model_dir, "final_y.pth"))
torch.save(u.state_dict(), os.path.join(model_dir, "final_u.pth"))
torch.save(theta.state_dict(), os.path.join(model_dir, "final_theta.pth"))

## After bilevel training, fix u and solve the lower-level energy problem separately to obtain an exact y

In [ ]:
def clone_trainer_and_train_fixed_v(original_trainer, additional_epochs=5000):
    """
    Create a clone of the trainer and continue training with v fixed, optimizing y using J_l loss.
    
    Args:
        original_trainer: The original trained Trainer instance
        additional_epochs: Number of additional epochs for the fixed-v training
        
    Returns:
        cloned_trainer: The cloned trainer instance after additional training
    """
    import copy
    import torch
    from tqdm import tqdm
    import time
    
    cloned_trainer = copy.copy(original_trainer)
    
    # Deep copy the networks (important to not affect original networks)
    cloned_trainer.y = copy.deepcopy(original_trainer.y)
    cloned_trainer.v = copy.deepcopy(original_trainer.u)
    cloned_trainer.theta = copy.deepcopy(original_trainer.theta) if original_trainer.theta is not None else None
    
    # Create fresh optimizers with the same parameters
    lr_y = original_trainer.optimizer_y.param_groups[0]['lr']
    cloned_trainer.optimizer_y = torch.optim.Adam(cloned_trainer.y.parameters(), lr=lr_y*0.005)
    
    # Copy scheduler configuration and create new scheduler
    if hasattr(original_trainer, 'scheduler_y') and original_trainer.scheduler_y is not None:
        from functools import partial
        from torch.optim.lr_scheduler import LambdaLR
        
        if hasattr(original_trainer, 'lr') and original_trainer.lr is not None:
            cloned_trainer.scheduler_y = LambdaLR(
                cloned_trainer.optimizer_y, 
                partial(lr_lambda, warmup=500, cycle_factor=0.8, cycle_length=1000)
            )

    
    # Set v to eval mode and freeze parameters
    cloned_trainer.v.eval()
    for param in cloned_trainer.v.parameters():
        param.requires_grad = False
    
    # Create a new file suffix to avoid overwriting original logs
    cloned_trainer.file_suffix = f"{original_trainer.file_suffix}_fixed_v_jl"
    
    # Start training loop with fixed v
    print(f"\nStarting additional training with fixed v for {additional_epochs} epochs...")
    print("Optimizing only J_l (energy functional) with fixed control")
    
    # Setup progress bar - now starting from 1
    pbar = tqdm(range(1, additional_epochs + 2), ascii=True, dynamic_ncols=True)
    cloned_trainer.source = original_trainer.source
    
    for epoch in pbar:
        cloned_trainer.y.train()
        
        total_loss = 0.0
        batch_count = 0
        
        for batch in cloned_trainer.train_loader:
            x = batch[0].detach().requires_grad_(True).to(cloned_trainer.device)
            
            # Only optimize for J_l loss
            loss_y = cloned_trainer.getloss_l(cloned_trainer.source, cloned_trainer.y, cloned_trainer.v, x)
            
            cloned_trainer._update_model(cloned_trainer.y, cloned_trainer.optimizer_y, loss_y)
            
            total_loss += loss_y.item()
            batch_count += 1
        
        if hasattr(cloned_trainer, 'scheduler_y') and cloned_trainer.scheduler_y is not None:
            cloned_trainer.scheduler_y.step()
        
        avg_loss = total_loss / batch_count if batch_count > 0 else 0
        
        # Record metrics periodically (without saving)
        if (epoch - 1) % 10 == 0:
            # Evaluate on grid for consistent metrics
            grid_size = 50
            x1 = torch.linspace(0, 1, grid_size, device=cloned_trainer.device)
            x2 = torch.linspace(0, 1, grid_size, device=cloned_trainer.device)
            X1, X2 = torch.meshgrid(x1, x2, indexing='ij')
            grid_points = torch.stack([X1.flatten(), X2.flatten()], dim=1).requires_grad_(True)
            
            
            # Clear cache periodically
            if epoch % 10 == 0 and torch.cuda.is_available():
                torch.cuda.empty_cache()
        
        pbar.set_postfix(
            J_l_loss=avg_loss
        )
        
        # Validate periodically (just calculate error, no plotting)
        if (epoch-1) % 500 == 0:
            cloned_trainer.y.eval()
            rel_l2_y, rel_l2_u = rel_l2_error_yu(cloned_trainer.y, cloned_trainer.v, cloned_trainer.source, device=cloned_trainer.device)
            visual_y_3d(cloned_trainer.source, cloned_trainer.y, file_name=None)  # No file saving
            if not hasattr(cloned_trainer, 'validation_csv_file'):
                validation_dir = os.path.join(ROOT, original_trainer.file_suffix)
                os.makedirs(validation_dir, exist_ok=True)
                cloned_trainer.validation_csv_file = os.path.join(validation_dir, 'forward_l2errors.csv')
                
                with open(cloned_trainer.validation_csv_file, 'w') as f:
                    f.write("epoch,rel_l2_error_y,rel_l2_error_u\n")
            
            with open(cloned_trainer.validation_csv_file, 'a') as f:
                f.write(f"{epoch},{rel_l2_y.item()},{rel_l2_u.item()}\n")
            
            
    cloned_trainer.num_epochs = additional_epochs
    
    return cloned_trainer

In [ ]:
cloned_trainer = clone_trainer_and_train_fixed_v(trainer, additional_epochs=5000)

torch.save(cloned_trainer.y.state_dict(), os.path.join(model_dir, "final_y_fixu.pth"))

In [ ]:
for name, param in cloned_trainer.y.named_parameters():
    print(f"Parameter name: {name}, Shape: {param.shape}")
cloned_trainer.y._psi

# Final plots

In [ ]:
def plot_solutions(net_y, net_u, source, device='cuda', save_path=None):
    """Visualize y/u on [0,1]×[0,1] and save only PDFs."""
    n_points = 200

    x = torch.linspace(0, 1, n_points, device=device)
    X, Y = torch.meshgrid(x, x, indexing='ij')
    xy = torch.cat((X.reshape(-1, 1), Y.reshape(-1, 1)), dim=1)

    with torch.no_grad():
        y_pred = net_y(xy).reshape(n_points, n_points).cpu()
        u_pred = net_u(xy).reshape(n_points, n_points).cpu()

    Xc = X.cpu().numpy()
    Yc = Y.cpu().numpy()
    y_pred_np = y_pred.numpy()
    u_pred_np = u_pred.numpy()
    has_analytic = hasattr(source, 'analystic') and source.analystic

    base_dir = None
    if save_path:
        base_dir = save_path
        os.makedirs(base_dir, exist_ok=True)

    def save_fig(name):
        if base_dir:
            plt.savefig(os.path.join(base_dir, f"{name}.pdf"), bbox_inches="tight")

    def add_colorbar(mappable):
        cbar = plt.colorbar(mappable, pad=0.1, fraction=0.046, shrink=1.0)
        cbar.ax.tick_params(labelsize=14)
        return cbar

    def stylize_3d(ax):
        ax.tick_params(axis='both', which='major', labelsize=14)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)

    def stylize_heatmap():
        plt.xticks(fontsize=14)
        plt.yticks(fontsize=14)

    print("Numerical Solution - State y")
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')
    surf = ax.plot_surface(Xc, Yc, y_pred_np, cmap='coolwarm')
    stylize_3d(ax)
    add_colorbar(surf)
    save_fig('example4_computed_y')
    plt.show()

    plt.close('all')
    print("Numerical Solution - Control u")
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')
    surf = ax.plot_surface(Xc, Yc, u_pred_np, cmap='coolwarm')
    stylize_3d(ax)
    add_colorbar(surf)
    save_fig('example4_computed_u')
    plt.show()

    if has_analytic:
        with torch.no_grad():
            y_star = source.compute_y_star(xy).reshape(n_points, n_points).cpu()
            u_star = source.compute_u_star(xy).reshape(n_points, n_points).cpu()
            diff_y = y_star - y_pred
            diff_u = u_star - u_pred

        y_star_np = y_star.numpy()
        u_star_np = u_star.numpy()
        diff_y_np = diff_y.numpy()
        diff_u_np = diff_u.numpy()

        plt.close('all')
        print("Exact Solution - State y")
        fig = plt.figure(figsize=(8, 6))
        ax = fig.add_subplot(111, projection='3d')
        surf = ax.plot_surface(Xc, Yc, y_star_np, cmap='coolwarm')
        stylize_3d(ax)
        add_colorbar(surf)
        save_fig('example4_y_exact')
        plt.show()

        plt.close('all')
        print("Exact Solution - Control u")
        fig = plt.figure(figsize=(8, 6))
        ax = fig.add_subplot(111, projection='3d')
        surf = ax.plot_surface(Xc, Yc, u_star_np, cmap='coolwarm')
        stylize_3d(ax)
        add_colorbar(surf)
        save_fig('example4_u_exact')
        plt.show()

        plt.close('all')
        print("Error |y - y*|")
        fig = plt.figure(figsize=(8, 6))
        ax = fig.add_subplot(111, projection='3d')
        surf = ax.plot_surface(Xc, Yc, diff_y_np, cmap='coolwarm')
        stylize_3d(ax)
        add_colorbar(surf)
        save_fig('example4_y_error')
        plt.show()

        plt.close('all')
        print("Error |u - u*|")
        fig = plt.figure(figsize=(8, 6))
        ax = fig.add_subplot(111, projection='3d')
        surf = ax.plot_surface(Xc, Yc, diff_u_np, cmap='coolwarm')
        stylize_3d(ax)
        add_colorbar(surf)
        save_fig('example4_u_error')
        plt.show()

        rel_l2_error_yu(net_y, net_u, source, device=device, resolution=101)

In [ ]:
# Replot using the fixu models
# Load the final checkpoints saved by the training cell above.
fixu_model_dir = os.path.join(ROOT, trainer.file_suffix, "models")
print(f"Loading fixu models from: {fixu_model_dir}")

net_u_fixu = net_u_res(source=source, hidden=16, layers=4, activation='Swish', hard_=hard_constraint_2d).to(device)
net_y_fixu = net_y_res(psi=net_u_fixu, source=source, hidden=16, layers=4, activation='Swish', hard_=hard_constraint_2d).to(device)
net_theta_fixu = net_y_res(psi=net_u_fixu, source=source, hidden=16, layers=4, activation='Swish', hard_=hard_constraint_2d).to(device)

net_y_fixu.load_state_dict(torch.load(os.path.join(fixu_model_dir, 'final_y_fixu.pth'), map_location=device))
net_u_fixu.load_state_dict(torch.load(os.path.join(fixu_model_dir, 'final_u.pth'), map_location=device))
net_theta_fixu.load_state_dict(torch.load(os.path.join(fixu_model_dir, 'final_theta.pth'), map_location=device))

net_y_fixu.eval()
net_u_fixu.eval()
net_theta_fixu.eval()

solution_dir_fixu = os.path.join(ROOT, 'solution_comparison_fixed_models')
os.makedirs(solution_dir_fixu, exist_ok=True)

print('Plotting with fixu models...')
plot_solutions(net_y_fixu, net_u_fixu, source, device=device, save_path=solution_dir_fixu)
